# Experiments: Goal-Driven Guide

This notebook shows how to use the `inference.experiments` API to run **prompt × model** experiments: define a grid of prompts and models, run all cells (with optional resume and retries), then analyse results. Each section is organised by **goal**, **when/why** it matters, and **how** to do it with code.

**Key concepts**
- **Raw DataFrame** (from `result.dataframe` or `build_dataframe_from_csv`): one row per prompt; columns are `prompt_id`, `prompt` (full prompt as stored), and one column per model. Each model cell is a **dict** with `status`, `response`, `error_message`, `metadata`. Use this when you need to inspect status, debug failures, or filter by completion.
- **Analysis DataFrame** (from `to_analysis_dataframe(raw_df)`): same rows, but the `prompt` column is the raw prompt string and each model column is **just the response text** (or `None` if the cell did not succeed). No status/error dicts—ideal for comparing answers across models or prompts.
- **Prompts** are stored in a single canonical format (messages-style JSON) so that system vs user vs multi-turn is consistent everywhere; you always get one `prompt` column you can search with `prompt_contains`.

Sections below: setup → first run → resume → grid design (system/context/per-model/sparse) → multi-model and retries → combined example → API reference.


## 1. Setup

**Goal:** Get ready to run experiments from this repo.

You need an **inference client** (configured with API keys and model endpoints) and the **experiments** API. The client is created from a config file (e.g. `config/inference.example.yaml`). The snippet below finds the config from either the repo root or the `examples/` directory so the notebook runs in both places. Imports: `ExperimentConfig`, `ExperimentRunner`, `build_experiment_grid`, `build_dataframe_from_csv`, `filter_experiment_dataframe`, `to_analysis_dataframe`.


In [1]:
from pathlib import Path

from inference import create_client
from inference.experiments import (
    ExperimentConfig,
    ExperimentResult,
    ExperimentRetryOptions,
    ExperimentRunner,
    ExperimentSchedulingOptions,
    build_dataframe_from_csv,
    build_experiment_grid,
    filter_experiment_dataframe,
    to_analysis_dataframe,
)


# Resolve config path so the notebook works from repo root or from examples/
def _repo_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "config" / "inference.example.yaml").exists():
            return p
    return Path.cwd()


CONFIG_PATH = _repo_root() / "config" / "inference.example.yaml"
client = create_client(CONFIG_PATH)

## 2. Run your first experiment

**Goal:** Run a small prompt × model matrix and inspect results.

**Workflow:** Build an `ExperimentConfig` with `experiment_name`, `model_aliases`, and `prompts`. The runner executes every (prompt, model) cell, writes results to a timestamped CSV under `logs/<experiment_name>/`, and returns an `ExperimentResult` with `.dataframe` (raw), `.csv_path`, and `.summary`. Use the raw DataFrame for debugging and filtering; use `to_analysis_dataframe(raw_df)` when you only need prompt + response text per model. Set `verbosity` to `"quiet"`, `"normal"`, `"verbose"`, or `"debug"` to control logging.


In [2]:
# Small matrix: 3 prompts × 1 model
config = ExperimentConfig(
    experiment_name="basic-demo",
    model_aliases=["gemma-3-4b"],
    prompts=[
        "What is 2+2?",
        "What is the capital of France?",
        "Name a primary color.",
    ],
    verbosity="verbose",  # "quiet" | "normal" | "verbose" | "debug"
)
runner = ExperimentRunner(client)
result: ExperimentResult = await runner.run(config)

Experiment: 3 prompts × 1 models = 3 cells
  TIMESTAMP            PROVIDER     MODEL                      STATUS    LATENCY PROGRESS
  2026-03-11 17:33:35  openrouter   google/gemma-3-4b-it:free  success     1.31s    (1/3)
  2026-03-11 17:33:35  openrouter   google/gemma-3-4b-it:free  success     1.99s    (2/3)
  2026-03-11 17:33:36  openrouter   google/gemma-3-4b-it:free  success     2.21s    (3/3)
Done.


In [ ]:
print("Summary:", result.summary.prompt_count, "prompts,", result.summary.model_count, "models,", result.summary.completed_cells, "/", result.summary.total_cells, "completed.")
print("CSV:", result.csv_path)

Summary: 3 prompts, 1 models, 3 / 3 completed.
CSV: logs/basic-demo/20260311T173333.csv


In [4]:
# Raw DataFrame: one row per prompt; model columns are cell dicts (status, response, error_message, metadata)
result.dataframe

,prompt_id,prompt,gemma-3-4b
0,d4e2520adda51c130e00051ed32c7576ade331818940c5...,"{""messages"":[{""content"":""What is 2+2?"",""role"":...","{'status': 'success', 'response': '2 + 2 = 4 '..."
1,c574617fe19e88afc543f2a0287160c93a7de87a874a05...,"{""messages"":[{""content"":""What is the capital o...","{'status': 'success', 'response': 'The capital..."
2,72a436f7a0dd7a7db010e96e24614cb35de298dea40478...,"{""messages"":[{""content"":""Name a primary color....","{'status': 'success', 'response': 'Red! 😊 Wo..."


In [5]:
# Analysis DataFrame: same rows, but model columns are just response text (or None if not successful)—best for comparing answers
to_analysis_dataframe(result.dataframe).head()

,prompt_id,prompt,gemma-3-4b
0,d4e2520adda51c130e00051ed32c7576ade331818940c5...,"{""messages"":[{""content"":""What is 2+2?"",""role"":...",2 + 2 = 4\n
1,c574617fe19e88afc543f2a0287160c93a7de87a874a05...,"{""messages"":[{""content"":""What is the capital o...",The capital of France is **Paris**. \n\nIt’s a...
2,72a436f7a0dd7a7db010e96e24614cb35de298dea40478...,"{""messages"":[{""content"":""Name a primary color....",Red! 😊 \n\nWould you like to know more about p...


## 3. Resume and extend runs

**Goal:** After a crash or interruption, continue without re-running completed cells; or add more prompts and run only the new cells.

**When to use:** Long experiments, rate limits, or when you want to extend an existing run with more prompts. Set `resume_from_existing_csv=True`; the runner finds the latest CSV for that `experiment_name`, skips cells that are already successful, and runs only missing/failed/rate-limited ones. Adding prompts appends new rows and runs only those new cells; passing fewer prompts trims the CSV to the current set. To point to a specific CSV file instead of auto-resolving, set `existing_csv_path=Path("...")`.


In [6]:
# Same experiment name, more prompts → only new cells are run
config_extend = ExperimentConfig(
    experiment_name="basic-demo",
    model_aliases=["gemma-3-4b"],
    prompts=[
        "What is 2+2?",
        "What is the capital of France?",
        "Name a primary color.",
        "What is the speed of light?",
        "Who wrote Romeo and Juliet?",
    ],
    resume_from_existing_csv=True,
)
extended_result: ExperimentResult = await runner.run(config_extend)

Resume existing run: 5 prompts × 1 models = 2 cells
  TIMESTAMP            PROVIDER     MODEL                      STATUS    LATENCY PROGRESS
  2026-03-11 17:33:37  openrouter   google/gemma-3-4b-it:free  success     1.68s    (1/2)
  2026-03-11 17:33:42  openrouter   google/gemma-3-4b-it:free  success     6.21s    (2/2)
Done.


In [7]:
# Load results from disk (e.g. in another session)—same raw shape: prompt_id, prompt, model columns
df = build_dataframe_from_csv(extended_result.csv_path)
print("Raw columns:", list(df.columns))
df.head()

Raw columns: ['prompt_id', 'prompt', 'gemma-3-4b']


,prompt_id,prompt,gemma-3-4b
0,d4e2520adda51c130e00051ed32c7576ade331818940c5...,"{""messages"":[{""content"":""What is 2+2?"",""role"":...","{'status': 'success', 'response': '2 + 2 = 4 '..."
1,c574617fe19e88afc543f2a0287160c93a7de87a874a05...,"{""messages"":[{""content"":""What is the capital o...","{'status': 'success', 'response': 'The capital..."
2,72a436f7a0dd7a7db010e96e24614cb35de298dea40478...,"{""messages"":[{""content"":""Name a primary color....","{'status': 'success', 'response': 'Red! 😊 Wo..."
3,c1403ed7b40905dac7ad6cc43e1ffad6542363bbcfb53a...,"{""messages"":[{""content"":""What is the speed of ...","{'status': 'success', 'response': 'The speed o..."
4,311b9b89d3e49b71c3ba78e5641609ffb334a70f4732c7...,"{""messages"":[{""content"":""Who wrote Romeo and J...","{'status': 'success', 'response': 'William Sha..."


## 4. Design experiment grids

**Goal:** Build the **row dimension** of your experiment (prompts) from combinations of system instructions, conversation context, and final user messages—so you don't hand-build every prompt.

Every experiment is a **matrix**: rows = prompts, columns = models. `build_experiment_grid` generates the list of prompts (and optional `run_cells`) from high-level inputs. You then pass `grid.prompts` and `grid.model_aliases` into `ExperimentConfig`. Below: (4a) system × request, (4b) multi-turn context, (4c) per-model system prompt, (4d) run only a subset of cells. Some providers fold the system prompt into the user message; you can check `metadata.system_prompt_folded` on cells when analysing.


### 4a. Compare instruction style (system × request)

Same user questions under **different system prompts**. Useful for studying how instruction style affects answers (e.g. "be concise" vs "be thorough").


In [9]:
# Cartesian product: each (system, user_message) becomes one prompt row
grid_sr = build_experiment_grid(
    system_prompts=[
        "You are concise. Answer in one short sentence.",
        "You are thorough. Explain briefly.",
    ],
    final_user_messages=["What is 2+2?", "What is the capital of France?"],
    model_aliases=["gemma-3-4b"],
)
config_sr = ExperimentConfig(
    experiment_name="grid-style-demo",
    model_aliases=grid_sr.model_aliases,
    prompts=grid_sr.prompts,
)
result_sr: ExperimentResult = await runner.run(config_sr)
# Analysis view: prompt (full) + response text per model
analysis_sr = to_analysis_dataframe(result_sr.dataframe)
analysis_sr[["prompt", "gemma-3-4b"]].head()

Experiment: 4 prompts × 1 models = 4 cells
  TIMESTAMP            PROVIDER     MODEL                      STATUS    LATENCY PROGRESS
  2026-03-11 17:33:43  openrouter   google/gemma-3-4b-it:free  success     1.00s    (1/4)
  2026-03-11 17:33:43  openrouter   google/gemma-3-4b-it:free  success     1.37s    (2/4)
  2026-03-11 17:33:44  openrouter   google/gemma-3-4b-it:free  success     1.55s    (3/4)
  2026-03-11 17:33:44  openrouter   google/gemma-3-4b-it:free  success     1.91s    (4/4)
Done.


,prompt,gemma-3-4b
0,"{""messages"":[{""content"":""You are concise. Answ...",2 + 2 = 4.
1,"{""messages"":[{""content"":""You are concise. Answ...",Paris is the capital of France.
2,"{""messages"":[{""content"":""You are thorough. Exp...",2 + 2 = 4\n\nIt’s a fundamental addition probl...
3,"{""messages"":[{""content"":""You are thorough. Exp...",The capital of France is **Paris**. \n\nIt’s t...


### 4b. Multi-turn (static context + varying final message)

Same **prior conversation** (system + `static_turns`) with multiple **final user messages**. For different conversation histories per row, use `static_turns_list` instead of `static_turns`.


In [10]:
# One system + one prior exchange, multiple follow-up questions
grid_mt = build_experiment_grid(
    system_prompt="You are a helpful math tutor. Answer briefly.",
    static_turns=[
        {"role": "user", "content": "What is 2+2?"},
        {"role": "assistant", "content": "2+2 equals 4."},
    ],
    final_user_messages=["So what is 2+2+2?", "What is 3+3?"],
    model_aliases=["gemma-3-4b"],
)
config_mt = ExperimentConfig(
    experiment_name="multi-turn-demo",
    model_aliases=grid_mt.model_aliases,
    prompts=grid_mt.prompts,
)
result_mt: ExperimentResult = await runner.run(config_mt)
analysis_mt = to_analysis_dataframe(result_mt.dataframe)
analysis_mt[["prompt", "gemma-3-4b"]].head()

Experiment: 2 prompts × 1 models = 2 cells
  TIMESTAMP            PROVIDER     MODEL                      STATUS    LATENCY PROGRESS
  2026-03-11 17:33:45  openrouter   google/gemma-3-4b-it:free  success     1.15s    (1/2)
  2026-03-11 17:33:45  openrouter   google/gemma-3-4b-it:free  success     1.49s    (2/2)
Done.


,prompt,gemma-3-4b
0,"{""messages"":[{""content"":""You are a helpful mat...",2 + 2 + 2 = 6 \n\nLet me know if you’d like to...
1,"{""messages"":[{""content"":""You are a helpful mat...",3 + 3 = 6 \n\nDo you want to try another math ...


### 4c. Per-model system prompt

Same **user prompts** for all models, but each model gets a **different system instruction** via `system_prompt_by_model`. Use this to compare answers when each model is given a different persona or interface (e.g. "You are ChatGPT" vs "You are Claude").


In [11]:
system_prompt_by_model = {
    "gemma-3-4b": "You are ChatGPT. Be concise and friendly.",
    "step-3.5-flash": "You are Claude. Be thorough and precise.",
}
grid_pm = build_experiment_grid(
    system_prompt_by_model=system_prompt_by_model,
    final_user_messages=["What is 2+2?", "Explain recursion in one sentence."],
    model_aliases=["gemma-3-4b", "step-3.5-flash"],
)
config_pm = ExperimentConfig(
    experiment_name="per-model-interface-demo",
    model_aliases=grid_pm.model_aliases,
    prompts=grid_pm.prompts,
    system_prompt_by_model=system_prompt_by_model,
    scheduling=ExperimentSchedulingOptions(interleave_model_aliases=True),
)
result_pm: ExperimentResult = await runner.run(config_pm)
analysis_pm = to_analysis_dataframe(result_pm.dataframe)
analysis_pm[["prompt", "gemma-3-4b", "step-3.5-flash"]].head()

Experiment: 2 prompts × 2 models = 4 cells
  TIMESTAMP            PROVIDER     MODEL                      STATUS    LATENCY PROGRESS
  2026-03-11 17:33:47  openrouter   google/gemma-3-4b-it:free  success     1.26s    (1/4)
  2026-03-11 17:33:47  openrouter   google/gemma-3-4b-it:free  success     1.33s    (2/4)
  2026-03-11 17:33:47  openrouter   stepfun/step-3.5-flash:f.. success     1.64s    (3/4)
  2026-03-11 17:33:52  openrouter   stepfun/step-3.5-flash:f.. success     6.43s    (4/4)
Done.


,prompt,gemma-3-4b,step-3.5-flash
0,"{""messages"":[{""content"":""What is 2+2?"",""role"":...",4! 😊 \n,2 + 2 equals **4**.
1,"{""messages"":[{""content"":""Explain recursion in ...",Recursion is when a function calls itself with...,Recursion is a programming technique where a f...


### 4d. Run only some cells (sparse grid)

Pass **`run_cells`**: a set of `(prompt_id, model_alias)` pairs. Only those cells are run; all others are written as `not_requested`. Use this to run a subset of the grid (e.g. one prompt × one model). You need the canonical `prompt_id` for each prompt—use `compute_prompt_id(canonical_prompt_spec(p))` so IDs match the matrix.


In [12]:
from inference.experiments.csv_schema import canonical_prompt_spec, compute_prompt_id

grid_sparse = build_experiment_grid(
    system_prompts=["System A", "System B"],
    final_user_message="One question.",
    model_aliases=["gemma-3-4b"],
)
# Run only the first prompt × gemma-3-4b; other cell will be not_requested
run_cells = {
    (compute_prompt_id(canonical_prompt_spec(p)), "gemma-3-4b") for p in grid_sparse.prompts[:1]
}
config_sparse = ExperimentConfig(
    experiment_name="sparse-grid-demo",
    model_aliases=grid_sparse.model_aliases,
    prompts=grid_sparse.prompts,
    run_cells=run_cells,
)
sparse_result: ExperimentResult = await runner.run(config_sparse)
# Inspect status per cell: one row will be success, the other not_requested
cell_statuses = sparse_result.dataframe["gemma-3-4b"].apply(
    lambda c: c.get("status") if isinstance(c, dict) else None
)
print("Cell statuses:", list(cell_statuses))

Experiment: 2 prompts × 1 models = 1 cells
  TIMESTAMP            PROVIDER     MODEL                      STATUS    LATENCY PROGRESS
  2026-03-11 17:33:53  openrouter   google/gemma-3-4b-it:free  success     1.18s    (1/1)
Done.
Cell statuses: ['success', 'not_requested']


In [13]:
# Keep only rows that were actually run, then get analysis view
run_only = sparse_result.dataframe[
    sparse_result.dataframe["gemma-3-4b"].apply(
        lambda c: c.get("status") != "not_requested" if isinstance(c, dict) else False
    )
]
to_analysis_dataframe(run_only)

,prompt_id,prompt,gemma-3-4b
0,d63ba2722e2bfbad4be343f7313455b3310288bd60d32c...,"{""messages"":[{""content"":""System A"",""role"":""sys...",Okay! Please ask your question. I’m ready.


## 5. Compare models and tune behaviour

**Goal:** Run **multiple models** on the same prompts (one column per model) and control **retries** and **rate-limit** behaviour.

- **Multiple models:** Set `model_aliases` to a list; you get one column per model in the raw and analysis DataFrames. Use `to_analysis_dataframe(df)` for side-by-side response text.
- **Retries:** `ExperimentRetryOptions(max_retries=..., base_delay=..., max_delay=...)`—use when you see transient failures or rate limits.
- **Scheduling:** `ExperimentSchedulingOptions(interleave_model_aliases=..., max_retry_after_wait_seconds=...)`—interleave `True` runs prompt-by-prompt across models; `False` runs one model at a time. Increase `max_retry_after_wait_seconds` if the API returns long retry-after headers.


In [14]:
# Multi-model run with explicit retry and scheduling options
config_multi = ExperimentConfig(
    experiment_name="multi-model-demo",
    model_aliases=["gemma-3-4b", "step-3.5-flash"],
    prompts=["What is machine learning?", "Explain neural networks."],
    retry=ExperimentRetryOptions(max_retries=3, base_delay=1.0, max_delay=30.0),
    scheduling=ExperimentSchedulingOptions(
        interleave_model_aliases=True,
        max_retry_after_wait_seconds=300,
    ),
)
multi_result: ExperimentResult = await runner.run(config_multi)
multi_result.summary

Experiment: 2 prompts × 2 models = 4 cells
  TIMESTAMP            PROVIDER     MODEL                      STATUS    LATENCY PROGRESS
  2026-03-11 17:34:00  openrouter   stepfun/step-3.5-flash:f.. success     6.36s    (1/4)
  2026-03-11 17:34:09  openrouter   stepfun/step-3.5-flash:f.. success    15.71s    (2/4)
  2026-03-11 17:34:21  openrouter   google/gemma-3-4b-it:free  success    27.95s    (3/4)
  2026-03-11 17:34:26  openrouter   google/gemma-3-4b-it:free  success    33.17s    (4/4)
Done.


ExperimentSummary(prompt_count=2, model_count=2, total_cells=4, completed_cells=4, failed_cells=0, rate_limited_cells=0)

In [15]:
multi_result.dataframe

,prompt_id,prompt,gemma-3-4b,step-3.5-flash
0,d0b39013260594f655d7e64ec1201caafc02cc84032a34...,"{""messages"":[{""content"":""What is machine learn...","{'status': 'success', 'response': 'Okay, let's...","{'status': 'success', 'response': 'Machine lea..."
1,9626cb05a7e36027c721efff376160cc89506311454a0b...,"{""messages"":[{""content"":""Explain neural networ...","{'status': 'success', 'response': 'Okay, let's...","{'status': 'success', 'response': 'Excellent q..."


In [22]:
# Analysis shape for side-by-side model comparison (response text per model)
to_analysis_dataframe(multi_result.dataframe).head().drop(columns=["prompt_id"])

,prompt,gemma-3-4b,step-3.5-flash
0,"{""messages"":[{""content"":""What is machine learn...","Okay, let's break down what machine learning i...",Machine learning is a branch of artificial int...
1,"{""messages"":[{""content"":""Explain neural networ...","Okay, let's break down neural networks. They'r...",Excellent question! Let's break down neural ne...


## 6. Combined full-featured example

**Goal:** Run one experiment that combines **several grid dimensions** (system prompts, final messages, static turn variants, per-model system, multiple models), then **filter** and **transform** to analysis shape.

**Data flow:** Run → **raw DataFrame** (prompt_id, prompt, one column per model with cell dicts) → optionally **filter** (e.g. `all_complete=True`, `prompt_contains="..."`, `models=[...]`, `min_success_per_row=...`) → **to_analysis_dataframe** → DataFrame with prompt + response text per model (or `None` where the cell didn't succeed). Filter kwargs can be passed directly into `to_analysis_dataframe(raw_df, prompt_contains="...")` so you don't need a separate filter step unless you want to reuse the filtered raw DataFrame.


In [23]:
# Grid: 2 systems × 2 final messages × 2 static turn variants × 2 models (each with its own system prompt)
system_prompt_by_model_full = {
    "gemma-3-4b": "Persona A: be brief.",
    "step-3.5-flash": "Persona B: be precise.",
}
grid_full = build_experiment_grid(
    system_prompts=["You are concise.", "You are thorough."],
    final_user_messages=["What is 2+2?", "What is the capital of France?"],
    static_turns_list=[
        [
            {"role": "user", "content": "Hi."},
            {"role": "assistant", "content": "Hello!"},
        ],
        [
            {"role": "user", "content": "Previous: 1+1=2."},
            {"role": "assistant", "content": "Correct."},
        ],
    ],
    system_prompt_by_model=system_prompt_by_model_full,
    model_aliases=["gemma-3-4b", "step-3.5-flash"],
)
config_full = ExperimentConfig(
    experiment_name="combined-demo",
    model_aliases=grid_full.model_aliases,
    prompts=grid_full.prompts,
    system_prompt_by_model=system_prompt_by_model_full,
    verbosity="quiet",
)
result_full: ExperimentResult = await runner.run(config_full)

In [24]:
# Raw DataFrame: prompt_id, prompt (full), then one column per model—each cell is a dict
raw_df = result_full.dataframe
print("Raw columns:", list(raw_df.columns))
raw_df.head()

Raw columns: ['prompt_id', 'prompt', 'gemma-3-4b', 'step-3.5-flash']


,prompt_id,prompt,gemma-3-4b,step-3.5-flash
0,7d703a0912c19c2dd287f2a89f67dfb020188501c3eb77...,"{""messages"":[{""content"":""Hi."",""role"":""user""},{...","{'status': 'success', 'response': '2 + 2 = 4 ...","{'status': 'success', 'response': '2 + 2 = 4',..."
1,e637409cfc76381fedab12e4c3e426bdd4950ea63319c2...,"{""messages"":[{""content"":""Hi."",""role"":""user""},{...","{'status': 'success', 'response': 'The capital...","{'status': 'success', 'response': 'The capital..."
2,ea4b5bcb573b30b099c6522ea6d09d213fff8da40a80e2...,"{""messages"":[{""content"":""Previous: 1+1=2."",""ro...","{'status': 'success', 'response': '2 + 2 = 4',...","{'status': 'success', 'response': '2+2 = 4.', ..."
3,6c5a027e65bd7c41666638daa5074de8baff9dcd22d2ab...,"{""messages"":[{""content"":""Previous: 1+1=2."",""ro...","{'status': 'success', 'response': 'The capital...","{'status': 'success', 'response': 'The capital..."


In [19]:
# Filter to rows where all models completed, then analysis view (prompt + response text only)
filtered = filter_experiment_dataframe(raw_df, all_complete=True)
analysis_df = to_analysis_dataframe(filtered)
print("Analysis columns:", list(analysis_df.columns))
analysis_df.head()

Analysis columns: ['prompt_id', 'prompt', 'gemma-3-4b', 'step-3.5-flash']


,prompt_id,prompt,gemma-3-4b,step-3.5-flash
0,7d703a0912c19c2dd287f2a89f67dfb020188501c3eb77...,"{""messages"":[{""content"":""Hi."",""role"":""user""},{...",2 + 2 = 4 \n\nDo you want to try another math ...,2 + 2 equals **4**.
1,e637409cfc76381fedab12e4c3e426bdd4950ea63319c2...,"{""messages"":[{""content"":""Hi."",""role"":""user""},{...",The capital of France is **Paris**. 😊 \n\nDo y...,The capital of France is **Paris**.
2,ea4b5bcb573b30b099c6522ea6d09d213fff8da40a80e2...,"{""messages"":[{""content"":""Previous: 1+1=2."",""ro...",2 + 2 = 4,2 + 2 = 4.
3,6c5a027e65bd7c41666638daa5074de8baff9dcd22d2ab...,"{""messages"":[{""content"":""Previous: 1+1=2."",""ro...",The capital of France is Paris.,The capital of France is **Paris**.


In [20]:
# Filter by prompt content: pass filter kwargs into to_analysis_dataframe
analysis_subset = to_analysis_dataframe(raw_df, prompt_contains="2+2")
analysis_subset[["prompt", "gemma-3-4b", "step-3.5-flash"]].head()

,prompt,gemma-3-4b,step-3.5-flash
0,"{""messages"":[{""content"":""Hi."",""role"":""user""},{...",2 + 2 = 4 \n\nDo you want to try another math ...,2 + 2 equals **4**.
1,"{""messages"":[{""content"":""Previous: 1+1=2."",""ro...",2 + 2 = 4,2 + 2 = 4.


In [21]:
# Raw + filter when you need status/errors (e.g. debugging); analysis when you need clean response text
filter_experiment_dataframe(raw_df, models=["gemma-3-4b"]).head()

,prompt_id,prompt,gemma-3-4b
0,7d703a0912c19c2dd287f2a89f67dfb020188501c3eb77...,"{""messages"":[{""content"":""Hi."",""role"":""user""},{...","{'status': 'success', 'response': '2 + 2 = 4 ..."
1,e637409cfc76381fedab12e4c3e426bdd4950ea63319c2...,"{""messages"":[{""content"":""Hi."",""role"":""user""},{...","{'status': 'success', 'response': 'The capital..."
2,ea4b5bcb573b30b099c6522ea6d09d213fff8da40a80e2...,"{""messages"":[{""content"":""Previous: 1+1=2."",""ro...","{'status': 'success', 'response': '2 + 2 = 4',..."
3,6c5a027e65bd7c41666638daa5074de8baff9dcd22d2ab...,"{""messages"":[{""content"":""Previous: 1+1=2."",""ro...","{'status': 'success', 'response': 'The capital..."


## 7. API reference

**Raw vs analysis:** The runner and `build_dataframe_from_csv` give you the **raw** DataFrame (prompt_id, prompt, model columns as cell dicts). For comparison and scripting, use **`to_analysis_dataframe(raw_df)`** to get prompt + response text per model (or `None`). Filter options: `filter_experiment_dataframe(raw_df, models=..., all_complete=..., prompt_contains=..., status=...)` or pass the same kwargs into `to_analysis_dataframe(raw_df, ...)`.

Compact reference for the main symbols. See the package docstring and `inference.experiments.types` for full signatures.

| Symbol | Type | Purpose |
|--------|------|---------|
| `ExperimentConfig` | dataclass | experiment_name, model_aliases, prompts; optional default_system_prompt, system_prompt_by_model, run_cells, retry, scheduling, verbosity, resume options |
| `ExperimentRunner` | class | `run(config)` → `ExperimentResult` |
| `ExperimentResult` | dataclass | dataframe, csv_path, csv_name, summary |
| `ExperimentSummary` | dataclass | prompt_count, model_count, total_cells, completed_cells, failed_cells, rate_limited_cells |
| `ExperimentRetryOptions` | dataclass | max_retries, base_delay, max_delay |
| `ExperimentSchedulingOptions` | dataclass | interleave_model_aliases, max_retry_after_wait_seconds |
| `VerbosityLevel` | type | `"quiet"` \| `"normal"` \| `"verbose"` \| `"debug"` |
| `build_dataframe_from_csv` | function | Load experiment CSV → raw DataFrame (prompt_id, prompt, model columns) |
| `filter_experiment_dataframe` | function | Filter raw DataFrame (models, all_complete, prompt_contains, status) |
| `to_analysis_dataframe` | function | Transform raw → analysis (prompt, response text per model); optional filter kwargs |
| `build_experiment_grid` | function | system/static/request modes → ExperimentGrid (prompts, model_aliases, run_cells) |
| `ExperimentGrid` | dataclass | prompts, model_aliases, run_cells |
| `PromptSpec` | type | str or dict (e.g. {"system": "...", "user": "..."}) |
